In [1]:
import pandas as pd
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# 1. 데이터셋 준비
data = {
    '성별': [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],  # 남 6명, 여 4명
    '결혼': [0, 0, 1, 1, 1, 0, 0, 0, 1, 1],  # 기혼 5명, 미혼 5명
    '타깃': [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]   # 파란색(0) 5개(충성고객), 빨간색(1) 5개(이탈고객)
}
df = pd.DataFrame(data)

# 피처(X)와 타깃(y) 분리
X = df[['성별', '결혼']]
y = df['타깃']

# 2. 개별 기저 모델 생성
lr_clf = LogisticRegression(random_state=42)
dt_clf = DecisionTreeClassifier(random_state=42)
knn_clf = KNeighborsClassifier(n_neighbors=3)

# 3. 하드 보팅 (Hard Voting) 모델 구축 및 학습
# 개별 모델들의 최종 예측 결과 중 다수결(Majority Vote)로 결정
hard_voting = VotingClassifier(
    estimators=[('LR', lr_clf), ('DT', dt_clf), ('KNN', knn_clf)],
    voting='hard'
)
hard_voting.fit(X, y)

# 4. 소프트 보팅 (Soft Voting) 모델 구축 및 학습
# 개별 모델들의 클래스별 예측 확률의 평균이 가장 높은 것을 선택
soft_voting = VotingClassifier(
    estimators=[('LR', lr_clf), ('DT', dt_clf), ('KNN', knn_clf)],
    voting='soft'
)
soft_voting.fit(X, y)

# 5. 결과 예측 및 출력
df['Hard_Pred'] = hard_voting.predict(X)
df['Soft_Pred'] = soft_voting.predict(X)

print(df)

# 소프트 보팅의 각 샘플별 클래스 예측 확률 확인 (0일 확률, 1일 확률)
soft_probs = soft_voting.predict_proba(X)
print("\n[소프트 보팅 예측 확률 산출 결과]")
for i, prob in enumerate(soft_probs):
    print(f"샘플 {i}: 클래스 0 확률={prob[0]:.2f}, 클래스 1 확률={prob[1]:.2f} -> 최종 예측={df['Soft_Pred'][i]}")

   성별  결혼  타깃  Hard_Pred  Soft_Pred
0   0   0   0          0          0
1   0   0   0          0          0
2   0   1   0          0          0
3   0   1   0          0          0
4   0   1   0          0          0
5   0   0   1          0          0
6   1   0   1          1          1
7   1   0   1          1          1
8   1   1   1          1          1
9   1   1   1          1          1

[소프트 보팅 예측 확률 산출 결과]
샘플 0: 클래스 0 확률=0.64, 클래스 1 확률=0.36 -> 최종 예측=0
샘플 1: 클래스 0 확률=0.64, 클래스 1 확률=0.36 -> 최종 예측=0
샘플 2: 클래스 0 확률=0.89, 클래스 1 확률=0.11 -> 최종 예측=0
샘플 3: 클래스 0 확률=0.89, 클래스 1 확률=0.11 -> 최종 예측=0
샘플 4: 클래스 0 확률=0.89, 클래스 1 확률=0.11 -> 최종 예측=0
샘플 5: 클래스 0 확률=0.64, 클래스 1 확률=0.36 -> 최종 예측=0
샘플 6: 클래스 0 확률=0.21, 클래스 1 확률=0.79 -> 최종 예측=1
샘플 7: 클래스 0 확률=0.21, 클래스 1 확률=0.79 -> 최종 예측=1
샘플 8: 클래스 0 확률=0.23, 클래스 1 확률=0.77 -> 최종 예측=1
샘플 9: 클래스 0 확률=0.23, 클래스 1 확률=0.77 -> 최종 예측=1
